In [8]:
options(stringsAsFactors = FALSE)

# =========================
# 0. 加载包
# =========================
library(data.table)
library(dplyr)
library(ggplot2)
library(stringr)
library(forcats)
library(fgsea)
library(msigdbr)

# =========================
# 1. 设置输入文件和输出目录
# =========================
file_golgi <- "golgi_tumbler_vs_homing_degs.csv"
file_purkinje <- "purkinje cells_tumbler_vs_homing_degs.csv"

outdir <- "GSEA_results_paper"
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)

# 物种
# human: "Homo sapiens"
# mouse: "Mus musculus"
species_use <- "Homo sapiens"

# 排序指标
rank_col_use <- "stat"

# 主图参数（更保守）
padj_cutoff_main <- 0.05
top_n_bar_main <- 10
top_n_compare_main <- 20
top_n_curve_each_direction <- 2   # 正NES取2个，负NES取2个

# 补充图参数（稍宽松）
padj_cutoff_supp <- 0.25
top_n_bar_supp <- 15
top_n_compare_supp <- 30

# 各基因集参数
param_list <- list(
  Hallmark = list(minSize = 10, maxSize = 500),
  GO_BP = list(minSize = 10, maxSize = 1000),
  Reactome = list(minSize = 10, maxSize = 500)
)

# =========================
# 2. 读取 DEG 文件
#    先按 abs(rank_col) 排，再去重
# =========================
read_deg_for_gsea <- function(file, rank_col = "stat") {
  df <- fread(file)
  colnames(df)[1] <- "gene"

  df <- df %>%
    filter(!is.na(gene), gene != "") %>%
    filter(!is.na(.data[[rank_col]])) %>%
    arrange(desc(abs(.data[[rank_col]]))) %>%
    distinct(gene, .keep_all = TRUE)

  return(df)
}

# =========================
# 3. 构建 preranked 排序向量
# =========================
make_rank_vector <- function(df, rank_col = "stat", jitter_ties = TRUE, seed = 123) {
  ranks <- df[[rank_col]]
  names(ranks) <- df$gene
  ranks <- ranks[!is.na(ranks)]

  if (jitter_ties) {
    set.seed(seed)
    ranks <- ranks + runif(length(ranks), -1e-10, 1e-10)
  }

  ranks <- sort(ranks, decreasing = TRUE)
  return(ranks)
}

# =========================
# 4. 准备基因集
# =========================
prepare_pathways <- function(species_use = "Homo sapiens") {
  msig_hallmark <- msigdbr(species = species_use, category = "H")
  pathways_hallmark <- split(msig_hallmark$gene_symbol, msig_hallmark$gs_name)

  msig_go_bp <- msigdbr(species = species_use, category = "C5", subcategory = "GO:BP")
  pathways_go_bp <- split(msig_go_bp$gene_symbol, msig_go_bp$gs_name)

  msig_reactome <- msigdbr(species = species_use, category = "C2", subcategory = "CP:REACTOME")
  pathways_reactome <- split(msig_reactome$gene_symbol, msig_reactome$gs_name)

  list(
    Hallmark = pathways_hallmark,
    GO_BP = pathways_go_bp,
    Reactome = pathways_reactome
  )
}

# =========================
# 5. overlap 检查
# =========================
check_overlap <- function(ranks, pathways, label = "", min_ratio_warn = 0.10) {
  pathway_genes <- unique(unlist(pathways))
  overlap_n <- sum(names(ranks) %in% pathway_genes)
  overlap_ratio <- overlap_n / length(ranks)

  overlap_df <- data.frame(
    label = label,
    total_ranked_genes = length(ranks),
    overlapping_genes = overlap_n,
    overlap_ratio = overlap_ratio
  )

  cat("\n============================\n")
  cat(label, "\n")
  cat("Total ranked genes :", length(ranks), "\n")
  cat("Overlapping genes  :", overlap_n, "\n")
  cat("Overlap ratio      :", round(overlap_ratio, 4), "\n")

  if (overlap_ratio < min_ratio_warn) {
    warning(paste(label, "overlap ratio is low:", round(overlap_ratio, 4)))
  }

  return(overlap_df)
}

# =========================
# 6. 跑 fgsea
# =========================
run_fgsea_one <- function(ranks, pathways, minSize = 10, maxSize = 500) {
  fgseaRes <- fgsea(
    pathways = pathways,
    stats = ranks,
    minSize = minSize,
    maxSize = maxSize,
    eps = 0
  )

  fgseaRes <- fgseaRes %>%
    as.data.frame() %>%
    arrange(padj, desc(abs(NES)))

  return(fgseaRes)
}

# =========================
# 7. 保存 fgsea 结果表
# =========================
save_fgsea_csv <- function(fgseaRes, file) {
  df <- as.data.frame(fgseaRes)

  if ("leadingEdge" %in% colnames(df)) {
    df$leadingEdge <- sapply(df$leadingEdge, function(x) paste(x, collapse = ";"))
  }

  write.csv(df, file, row.names = FALSE)
}

# =========================
# 8. 通路名美化
# =========================
clean_pathway_name <- function(x, collection_name = NULL) {
  y <- x

  if (!is.null(collection_name)) {
    if (collection_name == "Hallmark") {
      y <- str_replace_all(y, "^HALLMARK_", "")
    } else if (collection_name == "GO_BP") {
      y <- str_replace_all(y, "^GOBP_", "")
    } else if (collection_name == "Reactome") {
      y <- str_replace_all(y, "^REACTOME_", "")
    }
  }

  y <- str_replace_all(y, "_", " ")
  return(y)
}

# =========================
# 9. 论文主图条形图
#    不再做NES兜底
# =========================
plot_gsea_bar_main <- function(fgseaRes,
                               title,
                               collection_name = NULL,
                               top_n = 10,
                               padj_cutoff = 0.05) {
  dfp <- fgseaRes %>%
    filter(!is.na(padj), padj < padj_cutoff)

  if (nrow(dfp) == 0) {
    message(title, ": no pathways with padj < ", padj_cutoff)
    return(NULL)
  }

  top_up <- dfp %>%
    filter(NES > 0) %>%
    arrange(padj, desc(NES)) %>%
    head(top_n)

  top_down <- dfp %>%
    filter(NES < 0) %>%
    arrange(padj, NES) %>%
    head(top_n)

  plot_df <- bind_rows(top_up, top_down) %>%
    distinct(pathway, .keep_all = TRUE)

  if (nrow(plot_df) == 0) return(NULL)

  plot_df$pathway_clean <- clean_pathway_name(plot_df$pathway, collection_name)

  p <- ggplot(plot_df, aes(x = NES, y = fct_reorder(pathway_clean, NES), fill = NES)) +
    geom_col(width = 0.75) +
    theme_bw(base_size = 12) +
    labs(
      title = title,
      x = "Normalized Enrichment Score (NES)",
      y = NULL
    ) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold"),
      axis.text.y = element_text(size = 10),
      panel.grid.major.y = element_blank()
    )

  return(p)
}

# =========================
# 10. 补充图条形图
#    可用较宽松阈值
# =========================
plot_gsea_bar_supp <- function(fgseaRes,
                               title,
                               collection_name = NULL,
                               top_n = 15,
                               padj_cutoff = 0.25) {
  dfp <- fgseaRes %>%
    filter(!is.na(padj), padj < padj_cutoff)

  if (nrow(dfp) == 0) {
    return(NULL)
  }

  top_up <- dfp %>%
    filter(NES > 0) %>%
    arrange(padj, desc(NES)) %>%
    head(top_n)

  top_down <- dfp %>%
    filter(NES < 0) %>%
    arrange(padj, NES) %>%
    head(top_n)

  plot_df <- bind_rows(top_up, top_down) %>%
    distinct(pathway, .keep_all = TRUE)

  if (nrow(plot_df) == 0) return(NULL)

  plot_df$pathway_clean <- clean_pathway_name(plot_df$pathway, collection_name)

  ggplot(plot_df, aes(x = NES, y = fct_reorder(pathway_clean, NES), fill = NES)) +
    geom_col(width = 0.75) +
    theme_bw(base_size = 12) +
    labs(
      title = title,
      x = "Normalized Enrichment Score (NES)",
      y = NULL
    ) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold"),
      axis.text.y = element_text(size = 9),
      panel.grid.major.y = element_blank()
    )
}

# =========================
# 11. enrichment curve
# =========================
plot_one_enrichment <- function(pathway_name, pathways, ranks, title = NULL) {
  if (!pathway_name %in% names(pathways)) {
    message("Pathway not found: ", pathway_name)
    return(NULL)
  }

  p <- plotEnrichment(pathways[[pathway_name]], ranks) +
    labs(
      title = ifelse(is.null(title), pathway_name, title),
      x = "Rank in ordered gene list",
      y = "Enrichment score"
    ) +
    theme_bw(base_size = 12) +
    theme(plot.title = element_text(face = "bold", hjust = 0.5))

  return(p)
}

# =========================
# 12. 论文主图比较图
#    只用显著通路，不做兜底
# =========================
plot_compare_dot_main <- function(gsea1,
                                  gsea2,
                                  label1 = "Golgi",
                                  label2 = "Purkinje",
                                  collection_name = NULL,
                                  top_n_compare = 20,
                                  padj_cutoff = 0.05,
                                  title = "GSEA comparison") {
  compare_df <- bind_rows(
    gsea1 %>% mutate(celltype = label1),
    gsea2 %>% mutate(celltype = label2)
  ) %>%
    filter(!is.na(padj), padj < padj_cutoff) %>%
    mutate(pathway_clean = clean_pathway_name(pathway, collection_name))

  if (nrow(compare_df) == 0) return(NULL)

  top_compare_pathways <- compare_df %>%
    group_by(pathway_clean) %>%
    summarise(best_padj = min(padj, na.rm = TRUE),
              best_absNES = max(abs(NES), na.rm = TRUE),
              .groups = "drop") %>%
    arrange(best_padj, desc(best_absNES)) %>%
    head(top_n_compare)

  plot_compare_df <- compare_df %>%
    filter(pathway_clean %in% top_compare_pathways$pathway_clean)

  pathway_order <- top_compare_pathways %>%
    arrange(best_absNES) %>%
    pull(pathway_clean)

  plot_compare_df$pathway_clean <- factor(plot_compare_df$pathway_clean, levels = pathway_order)

  p <- ggplot(
    plot_compare_df,
    aes(
      x = celltype,
      y = pathway_clean,
      size = -log10(padj + 1e-300),
      color = NES
    )
  ) +
    geom_point() +
    theme_bw(base_size = 12) +
    labs(
      title = title,
      x = NULL,
      y = NULL,
      size = "-log10(padj)",
      color = "NES"
    ) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold"),
      axis.text.y = element_text(size = 10)
    )

  return(p)
}

# =========================
# 13. 输出主图和补充分析
# =========================
run_and_export_collection_paper <- function(collection_name,
                                            pathways,
                                            ranks_golgi,
                                            ranks_purkinje,
                                            outdir,
                                            minSize_use,
                                            maxSize_use,
                                            padj_cutoff_main = 0.05,
                                            padj_cutoff_supp = 0.25,
                                            top_n_bar_main = 10,
                                            top_n_bar_supp = 15,
                                            top_n_curve_each_direction = 2,
                                            top_n_compare_main = 20,
                                            top_n_compare_supp = 30) {

  cat("\n\n########################################\n")
  cat("Running collection:", collection_name, "\n")
  cat("########################################\n")

  collection_dir <- file.path(outdir, collection_name)
  dir.create(collection_dir, showWarnings = FALSE, recursive = TRUE)

  # 1. overlap
  overlap_golgi <- check_overlap(ranks_golgi, pathways, paste("Golgi", collection_name, "overlap"))
  overlap_purkinje <- check_overlap(ranks_purkinje, pathways, paste("Purkinje", collection_name, "overlap"))
  overlap_all <- bind_rows(overlap_golgi, overlap_purkinje)
  write.csv(overlap_all,
            file.path(collection_dir, paste0(collection_name, "_overlap_summary.csv")),
            row.names = FALSE)

  # 2. GSEA
  gsea_golgi <- run_fgsea_one(ranks_golgi, pathways, minSize = minSize_use, maxSize = maxSize_use)
  gsea_purkinje <- run_fgsea_one(ranks_purkinje, pathways, minSize = minSize_use, maxSize = maxSize_use)

  # 3. 保存全集结果
  save_fgsea_csv(gsea_golgi, file.path(collection_dir, paste0("Golgi_", collection_name, "_GSEA_all.csv")))
  save_fgsea_csv(gsea_purkinje, file.path(collection_dir, paste0("Purkinje_", collection_name, "_GSEA_all.csv")))

  # 4. 保存主结果表
  sig_main_golgi <- gsea_golgi %>% filter(!is.na(padj), padj < padj_cutoff_main)
  sig_main_purkinje <- gsea_purkinje %>% filter(!is.na(padj), padj < padj_cutoff_main)

  save_fgsea_csv(sig_main_golgi,
                 file.path(collection_dir, paste0("Golgi_", collection_name, "_GSEA_main_padj0.05.csv")))
  save_fgsea_csv(sig_main_purkinje,
                 file.path(collection_dir, paste0("Purkinje_", collection_name, "_GSEA_main_padj0.05.csv")))

  # 5. 保存补充结果表
  sig_supp_golgi <- gsea_golgi %>% filter(!is.na(padj), padj < padj_cutoff_supp)
  sig_supp_purkinje <- gsea_purkinje %>% filter(!is.na(padj), padj < padj_cutoff_supp)

  save_fgsea_csv(sig_supp_golgi,
                 file.path(collection_dir, paste0("Golgi_", collection_name, "_GSEA_supp_padj0.25.csv")))
  save_fgsea_csv(sig_supp_purkinje,
                 file.path(collection_dir, paste0("Purkinje_", collection_name, "_GSEA_supp_padj0.25.csv")))

  # 6. 主图条形图
  p_golgi_main <- plot_gsea_bar_main(
    gsea_golgi,
    title = paste("Golgi:", collection_name),
    collection_name = collection_name,
    top_n = top_n_bar_main,
    padj_cutoff = padj_cutoff_main
  )

  p_purkinje_main <- plot_gsea_bar_main(
    gsea_purkinje,
    title = paste("Purkinje:", collection_name),
    collection_name = collection_name,
    top_n = top_n_bar_main,
    padj_cutoff = padj_cutoff_main
  )

  if (!is.null(p_golgi_main)) {
    ggsave(file.path(collection_dir, paste0("MAIN_Golgi_", collection_name, "_barplot.pdf")),
           p_golgi_main, width = 8, height = 6)
    ggsave(file.path(collection_dir, paste0("MAIN_Golgi_", collection_name, "_barplot.png")),
           p_golgi_main, width = 8, height = 6, dpi = 300)
  }

  if (!is.null(p_purkinje_main)) {
    ggsave(file.path(collection_dir, paste0("MAIN_Purkinje_", collection_name, "_barplot.pdf")),
           p_purkinje_main, width = 8, height = 6)
    ggsave(file.path(collection_dir, paste0("MAIN_Purkinje_", collection_name, "_barplot.png")),
           p_purkinje_main, width = 8, height = 6, dpi = 300)
  }

  # 7. 补充条形图
  p_golgi_supp <- plot_gsea_bar_supp(
    gsea_golgi,
    title = paste("Golgi:", collection_name, "(supplementary)"),
    collection_name = collection_name,
    top_n = top_n_bar_supp,
    padj_cutoff = padj_cutoff_supp
  )

  p_purkinje_supp <- plot_gsea_bar_supp(
    gsea_purkinje,
    title = paste("Purkinje:", collection_name, "(supplementary)"),
    collection_name = collection_name,
    top_n = top_n_bar_supp,
    padj_cutoff = padj_cutoff_supp
  )

  if (!is.null(p_golgi_supp)) {
    ggsave(file.path(collection_dir, paste0("SUPP_Golgi_", collection_name, "_barplot.pdf")),
           p_golgi_supp, width = 10, height = 8)
  }

  if (!is.null(p_purkinje_supp)) {
    ggsave(file.path(collection_dir, paste0("SUPP_Purkinje_", collection_name, "_barplot.pdf")),
           p_purkinje_supp, width = 10, height = 8)
  }

  # 8. enrichment curve：正负各取代表通路
  top_golgi_up <- gsea_golgi %>%
    filter(!is.na(padj), padj < padj_cutoff_main, NES > 0) %>%
    arrange(padj, desc(NES)) %>%
    head(top_n_curve_each_direction) %>%
    pull(pathway)

  top_golgi_down <- gsea_golgi %>%
    filter(!is.na(padj), padj < padj_cutoff_main, NES < 0) %>%
    arrange(padj, NES) %>%
    head(top_n_curve_each_direction) %>%
    pull(pathway)

  top_purkinje_up <- gsea_purkinje %>%
    filter(!is.na(padj), padj < padj_cutoff_main, NES > 0) %>%
    arrange(padj, desc(NES)) %>%
    head(top_n_curve_each_direction) %>%
    pull(pathway)

  top_purkinje_down <- gsea_purkinje %>%
    filter(!is.na(padj), padj < padj_cutoff_main, NES < 0) %>%
    arrange(padj, NES) %>%
    head(top_n_curve_each_direction) %>%
    pull(pathway)

  top_pathways_golgi <- unique(c(top_golgi_up, top_golgi_down))
  top_pathways_purkinje <- unique(c(top_purkinje_up, top_purkinje_down))

  for (pw in top_pathways_golgi) {
    p <- plot_one_enrichment(
      pathway_name = pw,
      pathways = pathways,
      ranks = ranks_golgi,
      title = paste("Golgi:", clean_pathway_name(pw, collection_name))
    )
    if (!is.null(p)) {
      fn <- str_replace_all(pw, "[^A-Za-z0-9_]", "_")
      ggsave(file.path(collection_dir, paste0("MAIN_Golgi_", collection_name, "_", fn, "_enrichment.pdf")),
             p, width = 6, height = 4.5)
    }
  }

  for (pw in top_pathways_purkinje) {
    p <- plot_one_enrichment(
      pathway_name = pw,
      pathways = pathways,
      ranks = ranks_purkinje,
      title = paste("Purkinje:", clean_pathway_name(pw, collection_name))
    )
    if (!is.null(p)) {
      fn <- str_replace_all(pw, "[^A-Za-z0-9_]", "_")
      ggsave(file.path(collection_dir, paste0("MAIN_Purkinje_", collection_name, "_", fn, "_enrichment.pdf")),
             p, width = 6, height = 4.5)
    }
  }

  # 9. 主图比较图
  p_compare_main <- plot_compare_dot_main(
    gsea1 = gsea_golgi,
    gsea2 = gsea_purkinje,
    label1 = "Golgi",
    label2 = "Purkinje",
    collection_name = collection_name,
    top_n_compare = top_n_compare_main,
    padj_cutoff = padj_cutoff_main,
    title = paste(collection_name, "comparison")
  )

  if (!is.null(p_compare_main)) {
    ggsave(file.path(collection_dir, paste0("MAIN_Golgi_vs_Purkinje_", collection_name, "_compare_dotplot.pdf")),
           p_compare_main, width = 7, height = 8)
    ggsave(file.path(collection_dir, paste0("MAIN_Golgi_vs_Purkinje_", collection_name, "_compare_dotplot.png")),
           p_compare_main, width = 7, height = 8, dpi = 300)
  }

  # 10. 控制台输出
  cat("\n===== Golgi", collection_name, "main pathways =====\n")
  print(gsea_golgi %>% filter(!is.na(padj), padj < padj_cutoff_main) %>% select(pathway, NES, pval, padj) %>% head(20))

  cat("\n===== Purkinje", collection_name, "main pathways =====\n")
  print(gsea_purkinje %>% filter(!is.na(padj), padj < padj_cutoff_main) %>% select(pathway, NES, pval, padj) %>% head(20))

  return(list(
    overlap = overlap_all,
    gsea_golgi = gsea_golgi,
    gsea_purkinje = gsea_purkinje
  ))
}

# =========================
# 14. 主流程
# =========================
deg_golgi <- read_deg_for_gsea(file_golgi, rank_col = rank_col_use)
deg_purkinje <- read_deg_for_gsea(file_purkinje, rank_col = rank_col_use)

ranks_golgi <- make_rank_vector(deg_golgi, rank_col = rank_col_use, jitter_ties = TRUE)
ranks_purkinje <- make_rank_vector(deg_purkinje, rank_col = rank_col_use, jitter_ties = TRUE)

pathway_list <- prepare_pathways(species_use = species_use)

all_results <- list()

for (collection_name in names(pathway_list)) {
  this_min <- param_list[[collection_name]]$minSize
  this_max <- param_list[[collection_name]]$maxSize

  all_results[[collection_name]] <- run_and_export_collection_paper(
    collection_name = collection_name,
    pathways = pathway_list[[collection_name]],
    ranks_golgi = ranks_golgi,
    ranks_purkinje = ranks_purkinje,
    outdir = outdir,
    minSize_use = this_min,
    maxSize_use = this_max,
    padj_cutoff_main = padj_cutoff_main,
    padj_cutoff_supp = padj_cutoff_supp,
    top_n_bar_main = top_n_bar_main,
    top_n_bar_supp = top_n_bar_supp,
    top_n_curve_each_direction = top_n_curve_each_direction,
    top_n_compare_main = top_n_compare_main,
    top_n_compare_supp = top_n_compare_supp
  )
}

save(
  deg_golgi, deg_purkinje,
  ranks_golgi, ranks_purkinje,
  pathway_list, all_results,
  file = file.path(outdir, "all_GSEA_results_paper.RData")
)

cat("\n\nAll analyses completed successfully.\n")
cat("Results saved in:", outdir, "\n")



########################################
Running collection: Hallmark 
########################################

Golgi Hallmark overlap 
Total ranked genes : 12530 
Overlapping genes  : 2709 
Overlap ratio      : 0.2162 

Purkinje Hallmark overlap 
Total ranked genes : 13999 
Overlapping genes  : 2946 
Overlap ratio      : 0.2104 

===== Golgi Hallmark main pathways =====
                                   pathway       NES         pval         padj
1       HALLMARK_OXIDATIVE_PHOSPHORYLATION -2.252946 3.600679e-11 1.800340e-09
2                  HALLMARK_MYC_TARGETS_V1 -1.824943 4.045654e-06 1.011413e-04
3                      HALLMARK_GLYCOLYSIS -1.668476 1.596380e-04 2.660633e-03
4                  HALLMARK_UV_RESPONSE_DN  1.673459 3.576083e-04 4.470103e-03
5 HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY -1.707484 6.448076e-03 4.605769e-02
6                 HALLMARK_MITOTIC_SPINDLE  1.462502 5.124096e-03 4.605769e-02
7                    HALLMARK_ADIPOGENESIS -1.444232 6.173666e-03 4.60